# Nimbus adaptation comparison: CUDA LoRA experiments
Select **Runtime → Change runtime type → T4 GPU**. This notebook runs training and same-backend baselines. Results do not exist until the cells finish. Llama 3.2 requires access to the model on Hugging Face; accept its license in your own account. Do not paste tokens into notebook source.

In [ ]:
!git clone --branch codex/midterm-experiments https://github.com/Somama12/llm-adaptation-comparison.git
%cd llm-adaptation-comparison
%pip install -r requirements-train.txt

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import torch
assert torch.cuda.is_available(), "Select a GPU runtime first"
print(torch.cuda.get_device_name())

## Baselines
Use the same Llama base model and NF4 backend as LoRA. Latency here should be compared within Colab, not directly against Mac timings. Runs save partial results after each question.

In [ ]:
import subprocess
subprocess.run(['python', '-m', 'experiments.run', '--backend', 'hf', '--model', 'meta-llama/Llama-3.2-3B-Instruct', '--methods', 'tfidf', 'embedding', 'fewshot', 'zero', '--output', 'results/colab_baselines.json'], check=True)


## Train on Version 1, evaluate initial and stale knowledge
Training uses 13 independently worded handbook examples. Test facts overlap by design; test question wording is held out. This is a tiny memorization pilot, not broad generalization evidence.

In [ ]:
import subprocess
subprocess.run(['python', '-m', 'finetune.train', '--version', 'v1', '--output', 'artifacts/lora-v1'], check=True)
subprocess.run(['python', '-m', 'experiments.run', '--backend', 'hf', '--model', 'meta-llama/Llama-3.2-3B-Instruct', '--methods', 'lora', '--adapter', 'artifacts/lora-v1', '--knowledge-versions', 'v1', '--output', 'results/lora_v1.json'], check=True)


## Update the Version 1 adapter with Version 2 content
This measures recovery after continued training. Keep the Version 1 adapter for the stale condition. Training budget is fixed in advance.

In [ ]:
import subprocess
subprocess.run(['python', '-m', 'finetune.train', '--version', 'v2', '--resume-adapter', 'artifacts/lora-v1', '--output', 'artifacts/lora-v2'], check=True)
subprocess.run(['python', '-m', 'experiments.run', '--backend', 'hf', '--model', 'meta-llama/Llama-3.2-3B-Instruct', '--methods', 'lora', '--adapter', 'artifacts/lora-v2', '--knowledge-versions', 'v2', '--output', 'results/lora_v2.json'], check=True)


In [ ]:
import subprocess
subprocess.run(['python', '-m', 'experiments.report', '--inputs', 'results/colab_baselines.json', 'results/lora_v1.json', 'results/lora_v2.json', '--output', 'docs/colab_results.md'], check=True)
from pathlib import Path
assert all(Path(p).exists() for p in ["results/colab_baselines.json", "results/lora_v1.json", "results/lora_v2.json"]), "A prior command failed; inspect its output"
import shutil
for version in ["v1", "v2"]:
    shutil.copy(f"artifacts/lora-{version}/training_metadata.json", f"results/training_{version}.json")
shutil.make_archive("colab_results", "zip", root_dir=".", base_dir="results")
from google.colab import files
files.download("colab_results.zip")
